In [39]:
import tkinter as tk
import random
import heapq
import time
from collections import deque


In [40]:
GRID_SIZE = 20
CELL_SIZE = 30

START = (0, 0)
GOALS = [(19, 19), (19, 0), (0, 19)]
FIRES = [
    (1, 0), (0, 11),
    (2, 8), (2, 15), (2, 19),
    (3, 9), (3, 14), (3, 18),
    (4, 19),
    (5, 4), (5, 6), (5, 10),
    (6, 1), (6, 12),
    (7, 1), (7, 2), (7, 14),
    (8, 4), (8, 15),
    (9, 5), (9, 7), (9, 12), (9, 17), (9, 18),
    (10, 1), (10, 6), (10, 8), (10, 17),
    (11, 12), (11, 17),
    (12, 7), (12, 8),
    (13, 1), (13, 19),
    (14, 10), (14, 11), (14, 14), (14, 15), (14, 19),
    (15, 5),
    (16, 5), (16, 13), (16, 19),
    (18, 10),
    (19, 4), (19, 7), (19, 12)
]

# Uniform Cost Search

In [41]:
def ucs_fire(nodes, start, goals, fires, cell_cost):
    pq = []
    heapq.heappush(pq, (cell_cost[start[0]][start[1]], start, [start]))
    visited = set()
    expanded = []

    while pq:
        cost, node, path = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        expanded.append(node)

        if node in goals:
            return path, cost, expanded

        for n in nodes[node]:
            if n not in fires and n not in visited:
                r, c = n
                heapq.heappush(pq, (cost + cell_cost[r][c], n, path + [n]))
    return None, None, expanded


# Depth-First Search

In [42]:
def dfs_fire(nodes, start, goals, fires):
    stack = [(start, [start])]
    visited = set()
    expanded = []

    while stack:
        node, path = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        expanded.append(node)

        if node in goals:
            return path, expanded

        for n in reversed(nodes[node]):
            if n not in visited and n not in fires:
                stack.append((n, path + [n]))

    return None, expanded


# Breadth-First Search

In [43]:
def bfs_simple(nodes, start, goals, fires):
    q = deque([(start, [start])])
    visited = {start}
    expanded = []

    while q:
        node, path = q.popleft()
        expanded.append(node)

        if node in goals:
            return path, expanded

        for n in nodes[node]:
            if n not in visited and n not in fires:
                visited.add(n)
                q.append((n, path + [n]))

    return None, expanded


# Depth Limited Search


In [44]:
def depth_limited_dfs(node, goal, limit, path, visited, expansion_order, nodes, cell_cost, cost):
    expansion_order.append(node)
    if node == goal:
        return path, cost
    if limit == 0:
        return None, None
    visited.add(node)
    for neigh in nodes[node]:
        if neigh not in visited:
            r, c = neigh
            result_path, result_cost = depth_limited_dfs(
                neigh, goal, limit-1, path+[neigh], visited, expansion_order, nodes, cell_cost, cost + cell_cost[r][c]
            )
            if result_path is not None:
                return result_path, result_cost
    return None, None


# iterative_deepening_dfs

In [45]:
def iterative_deepening_dfs(start, goal, nodes, cell_cost):
    max_depth = len(nodes)
    for depth in range(max_depth):
        visited = set()
        expansion_order = []
        result_path, result_cost = depth_limited_dfs(
            start, goal, depth, [start], visited, expansion_order, nodes, cell_cost, cell_cost[start[0]][start[1]]
        )
        if result_path is not None:
            return result_path, expansion_order, result_cost, len(expansion_order)
    return None, [], None, 0


# Heuristic

In [46]:
def heuristic_to_goals(node, goals):
    return min(abs(node[0]-g[0]) + abs(node[1]-g[1]) for g in goals)




# A star 


In [47]:
def astar_fire(start, goals, fires, cell_cost):
    pq = [(heuristic_to_goals(start, goals) + cell_cost[start[0]][start[1]],
           cell_cost[start[0]][start[1]], start, [start])]
    visited = set()
    expanded = []

    while pq:
        f, g, node, path = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        expanded.append(node)

        if node in goals:
            return path, g, expanded

        for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
            nr, nc = node[0]+dr, node[1]+dc
            if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE and (nr, nc) not in fires:
                ng = g + cell_cost[nr][nc]
                heapq.heappush(
                    pq,
                    (ng + heuristic_to_goals((nr,nc), goals), ng, (nr, nc), path + [(nr, nc)])
                )
    return None, None, expanded




# greedy


In [48]:
def greedy_fire(start, goals, fires, cell_cost):
    pq = [(heuristic_to_goals(start, goals), start, [start], cell_cost[start[0]][start[1]])]
    visited = set()
    expanded = []

    while pq:
        h, node, path, cost = heapq.heappop(pq)
        if node in visited:
            continue
        visited.add(node)
        expanded.append(node)

        if node in goals:
            return path, cost, expanded

        for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
            nr, nc = node[0]+dr, node[1]+dc
            if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE and (nr, nc) not in fires:
                heapq.heappush(
                    pq,
                    (heuristic_to_goals((nr, nc), goals),
                     (nr, nc),
                     path + [(nr, nc)],
                     cost + cell_cost[nr][nc])
                )
    return None, None, expanded



# Gui


In [ ]:
class EvacuationGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Emergency_Evacuation_planner")

        self.grid = [[0]*GRID_SIZE for _ in range(GRID_SIZE)]
        self.cells = []
        self.path = []
        self.index = 0
        self.expanded = []

        self.algo = tk.StringVar(value="UCS")
        self.cell_cost_random = [[random.randint(1,5) for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]

        self.cell_cost_one = [[1 for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        self.cell_cost = self.cell_cost_random  

        top = tk.Frame(root)
        top.pack()
        tk.Label(top, text="Algorithm:").pack(side="left")
        tk.OptionMenu(top, self.algo, "UCS", "DFS", "BFS", "A*", "Greedy", "IDDFS").pack(side="left")
        tk.Button(top, text="Start", command=self.start).pack(side="left")
        tk.Button(top, text="Clear", command=self.clear).pack(side="left")

        self.info = tk.Label(root, text="")
        self.info.pack()

        self.canvas = tk.Canvas(root, width=GRID_SIZE*CELL_SIZE, height=GRID_SIZE*CELL_SIZE)
        self.canvas.pack()

        self.init_fire()
        self.draw_grid()

    def init_fire(self):
        for (r, c) in FIRES:
            if (r, c) != START and (r, c) not in GOALS:
                self.grid[r][c] = 1

    def draw_grid(self):
     self.canvas.delete("all")
     self.cells = []


     for r in range(GRID_SIZE):
        row = []
        for c in range(GRID_SIZE):
            color = "white"
            if (r, c) == START:
                color = "purple"
            elif (r, c) in GOALS:
                color = "green"
            elif self.grid[r][c] == 1:
                color = "red"

            rect = self.canvas.create_rectangle(
                c*CELL_SIZE, r*CELL_SIZE,
                (c+1)*CELL_SIZE, (r+1)*CELL_SIZE,
                fill=color, outline="black"
            )


            row.append(rect)
        self.cells.append(row)

    def clear(self):
        self.path = []
        self.index = 0
        self.expanded = []
        self.draw_grid()
        self.info.config(text="")

    def build_nodes(self, fires):
        nodes = {}
        for r in range(GRID_SIZE):
            for c in range(GRID_SIZE):
                if (r, c) not in fires:
                    nodes[(r, c)] = []
                    for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]:
                        nr, nc = r+dr, c+dc
                        if 0 <= nr < GRID_SIZE and 0 <= nc < GRID_SIZE and (nr, nc) not in fires:
                            nodes[(r, c)].append((nr, nc))
        return nodes

    def start(self):
        fires = {(r, c) for r in range(GRID_SIZE) for c in range(GRID_SIZE) if self.grid[r][c] == 1}
        nodes = self.build_nodes(fires)

        algo = self.algo.get()
        if algo in ["UCS", "A*", "Greedy"]:
            cell_cost = self.cell_cost_random
        else: 
            cell_cost = self.cell_cost_one

        self.cell_cost = cell_cost  

        t0 = time.time()
        total_cost = None  
        if algo == "DFS":
            self.path, self.expanded = dfs_fire(nodes, START, GOALS, fires)
        elif algo == "BFS":
            self.path, self.expanded = bfs_simple(nodes, START, GOALS, fires)
        elif algo == "A*":
            self.path, total_cost, self.expanded = astar_fire(START, GOALS, fires, cell_cost)
        elif algo == "Greedy":
            self.path, total_cost, self.expanded = greedy_fire(START, GOALS, fires, cell_cost)
        elif algo == "IDDFS":
            self.path, self.expanded, total_cost, _ = iterative_deepening_dfs(START, GOALS[0], nodes, cell_cost)
        else:  
            self.path, total_cost, self.expanded = ucs_fire(nodes, START, GOALS, fires, cell_cost)

        self.draw_grid()
        self.animate_expansion()

        self.info.config(
            text=f"{algo} | Path Total Cost: {total_cost if total_cost is not None else 'N/A'} | Steps: {len(self.path) if self.path else 'N/A'} | Time: {time.time()-t0:.3f}s"
        )

    def animate_expansion(self, index=0):
        if index >= len(self.expanded):
            self.index = 0
            self.move()
            return

        r, c = self.expanded[index]
        if (r, c) != START and (r, c) not in GOALS:
            self.canvas.itemconfig(self.cells[r][c], fill="yellow")

        self.root.after(50, lambda: self.animate_expansion(index + 1))

    def move(self):
        if self.index >= len(self.path):
            for r, c in self.path:
                if (r, c) != START and (r, c) not in GOALS:
                    self.canvas.itemconfig(self.cells[r][c], fill="purple")
            return

        r, c = self.path[self.index]
        self.index += 1
        self.draw_grid()
        x1 = c*CELL_SIZE + 6
        y1 = r*CELL_SIZE + 6
        x2 = x1 + CELL_SIZE - 12
        y2 = y1 + CELL_SIZE - 12
        self.canvas.create_oval(x1, y1, x2, y2, fill="purple")
        self.root.after(200, self.move)


In [50]:

root = tk.Tk()
EvacuationGUI(root)
root.mainloop()
